# XGBoost 贝叶斯超参搜索 (T-30)

用 Optuna 在 walk-forward 的 OOS 段上最大化 IC 的均值。
搜索空间:`max_depth ∈ [3, 8]`, `learning_rate ∈ [0.01, 0.3]` (log),
`n_estimators ∈ [50, 300]`, `subsample ∈ [0.6, 1.0]`。

⚠️ 完整运行需要 `pip install -e .[research]`(xgboost + optuna)。

In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO / 'src'))
import pandas as pd, numpy as np
try:
    import optuna
    import xgboost as xgb
except ImportError as e:
    raise SystemExit('pip install -e .[research]') from e

In [ ]:
rng = np.random.default_rng(7)
dates = pd.bdate_range('2022-01-03', periods=800)
X = pd.DataFrame({
    'f1': rng.normal(0, 1, 800),
    'f2': rng.normal(0, 1, 800),
    'f3': rng.normal(0, 1, 800),
}, index=dates)
y = 0.04 * X['f1'] + 0.02 * X['f2'] + 0.005 * rng.normal(0, 1, 800)

In [ ]:
def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
    }
    from pit_market.backtest import WalkForwardConfig, walk_forward
    returns = pd.DataFrame({'fwd_return_1d': y}, index=dates)
    cfg = WalkForwardConfig(
        train_size=252, test_size=63, step=63,
        feature_cols=('f1', 'f2', 'f3'),
        min_train=126,
        model_factory=lambda: xgb.XGBRegressor(**params, verbosity=0),
    )
    res = walk_forward(X, returns, cfg)
    return res.aggregate_ic()

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10, show_progress_bar=False)
study.best_params